In [ ]:
# ============================================================
# @title Model C (without Aug)2
# CNN from scratch - 256x256 - WITHOUT augmentation
# FULL STATE CHECKPOINTING (local + Drive)
# ============================================================

import os
import json
import copy
import time
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# LOCAL_RUN_NAME = "cnn_scratch_256_without_aug_2"

LOCAL_RUN_NAME = "model_i3_cnn256_noAug_from_scratch"

MODEL_NAME = "cnn_scratch_256_without_aug_2"


BASE_DIR = Path.cwd().parent.parent.parent
LOCAL_DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR_BASE = BASE_DIR / "outputs" / "image_modeling" / LOCAL_RUN_NAME
MODEL_DIR_BASE = BASE_DIR / "data" / "models" / LOCAL_RUN_NAME

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR_BASE.mkdir(parents=True, exist_ok=True)

# =====================
# LOAD DATA
# =====================
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Local save dirs for this run
# ------------------------------------------------------------
# LOCAL_RUN_NAME = "cnn_scratch_128_with_aug_2"
LOCAL_RUN_NAME = "model_i1_cnn_128_moderateAug_from_scratch"

LOCAL_OUTPUT_ROOT = BASE_DIR / "outputs" / LOCAL_RUN_NAME
LOCAL_MODEL_ROOT = LOCAL_DATA_DIR / "models" / LOCAL_RUN_NAME

for d in [
    LOCAL_OUTPUT_ROOT, LOCAL_MODEL_ROOT,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Local output root:", LOCAL_OUTPUT_ROOT)
# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------
IMAGE_SIZE = 256
BATCH_SIZE = 64
NUM_WORKERS = 2
MAX_EPOCHS = 35
EARLY_STOPPING_PATIENCE = 10

# Set this to True if you want to resume from last checkpoint automatically
RESUME_TRAINING = False

# ------------------------------------------------------------
# Transforms
# ------------------------------------------------------------
train_transform_C = transforms.Compose([
    transforms.Resize((IMAGE_SIZE ,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform_C = transforms.Compose([
    transforms.Resize((IMAGE_SIZE , IMAGE_SIZE )),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class RakutenImageDataset(Dataset):
    """
    Simple image dataset for Rakuten product images.

    Each item returns:
    - transformed image tensor
    - integer class label

    Expected dataframe columns:
    - image_path_local
    - label_id
    """
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "image_path_local"]
        label = int(self.df.loc[idx, "label_id"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset_C = RakutenImageDataset(train_df, transform=train_transform_C)
val_dataset_C = RakutenImageDataset(val_df, transform=val_transform_C)

train_loader_C = DataLoader(
    train_dataset_C,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader_C = DataLoader(
    val_dataset_C,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ScratchCNN256(nn.Module):
    """
    CNN from scratch for 256x256 RGB images.

    Architecture:
    - 4 convolutional blocks with BatchNorm + ReLU + MaxPool
    - adaptive average pooling
    - small MLP classifier head
    """
    def __init__(self, num_classes=27):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model_C = ScratchCNN256(num_classes=len(label2id)).to(device)

# ------------------------------------------------------------
# Training setup
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_C.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------
LAST_CKPT_LOCAL = LOCAL_MODEL_ROOT / "last_checkpoint.pt"
BEST_CKPT_LOCAL = LOCAL_MODEL_ROOT / "best_checkpoint.pt"
BEST_WEIGHTS_LOCAL = LOCAL_MODEL_ROOT / "best_model_state_dict.pt"
HISTORY_LOCAL = LOCAL_OUTPUT_ROOT / "history.csv"
METADATA_LOCAL = LOCAL_OUTPUT_ROOT / "run_metadata.json"

# ------------------------------------------------------------
# Helper: save checkpoint
# ------------------------------------------------------------
def save_full_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    history,
    best_f1,
    best_epoch,
    model_name,
    y_true=None,
    y_pred=None
):
    """
    Save the full training state needed to resume training exactly.

    Saved content:
    - epoch
    - model state_dict
    - optimizer state_dict
    - scheduler state_dict
    - history
    - best validation F1 and epoch
    - model name
    - RNG states for reproducibility / resuming
    - optional last validation true/pred labels
    """
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "model_name": model_name,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "y_true_last_val": y_true,
        "y_pred_last_val": y_pred,
    }

    if torch.cuda.is_available():
        checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    torch.save(checkpoint, checkpoint_path)


# ------------------------------------------------------------
# Helper: copy file
# ------------------------------------------------------------
def copy_to_drive(local_path):
    shutil.copy2(local_path)


# ------------------------------------------------------------
# Helper: save history and metadata
# ------------------------------------------------------------
def save_training_artifacts(history, best_f1, best_epoch, total_minutes):
    """
    Save lightweight run artifacts:
    - history.csv
    - metadata json
    """
    history_df = pd.DataFrame(
        history,
        columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc", "train_f1", "val_f1"]
    )
    history_df.to_csv(HISTORY_LOCAL, index=False)
    copy_to_drive(HISTORY_LOCAL)

    metadata = {
        "model_name": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "best_epoch": int(best_epoch),
        "best_macro_f1": float(best_f1),
        "training_minutes": float(total_minutes),
    }

    with open(METADATA_LOCAL, "w") as f:
        json.dump(metadata, f, indent=2)
    copy_to_drive(METADATA_LOCAL)


# ------------------------------------------------------------
# Helper: load checkpoint
# ------------------------------------------------------------
def load_full_checkpoint(checkpoint_path, model, optimizer, scheduler, device):
    """
    Load a full checkpoint and restore training state.

    Returns:
    - start_epoch
    - history
    - best_f1
    - best_epoch
    """
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    # Restore RNG states if available
    if "torch_rng_state" in checkpoint:
        torch.set_rng_state(checkpoint["torch_rng_state"])
    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])

    start_epoch =  1
    history = checkpoint.get("history", [])
    best_f1 = checkpoint.get("best_f1", 0.0)
    best_epoch = checkpoint.get("best_epoch", 0)

    print(f"Resumed from checkpoint: {checkpoint_path}")
    print(f"Starting at epoch: {start_epoch}")
    print(f"Best F1 so far: {best_f1:.4f} at epoch {best_epoch}")

    return start_epoch, history, best_f1, best_epoch


# ------------------------------------------------------------
# Training / validation for one epoch
# ------------------------------------------------------------
def run_epoch(model, loader, optimizer=None):
    """
    Run one full epoch.

    If optimizer is provided -> training mode.
    If optimizer is None -> evaluation mode.

    Returns:
    - average loss
    - accuracy
    - macro F1
    - true labels
    - predicted labels
    """
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss = 0.0
    preds_all = []
    labels_all = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(labels_all, preds_all)
    epoch_f1 = f1_score(labels_all, preds_all, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, np.array(labels_all), np.array(preds_all)


# ------------------------------------------------------------
# Resume setup
# ------------------------------------------------------------
start_epoch = 1
history = []
best_f1 = 0.0
best_epoch = 0
epochs_without_improvement = 0

if RESUME_TRAINING and LAST_CKPT_LOCAL.exists():
    start_epoch, history, best_f1, best_epoch = load_full_checkpoint(
        LAST_CKPT_LOCAL,
        model_C,
        optimizer,
        scheduler,
        device
    )
    epochs_without_improvement = start_epoch - best_epoch - 1

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    train_loss, train_acc, train_f1, _, _ = run_epoch(model_C, train_loader_C, optimizer=optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = run_epoch(model_C, val_loader_C, optimizer=None)

    scheduler.step(val_f1)

    history.append([
        epoch,
        train_loss,
        val_loss,
        train_acc,
        val_acc,
        train_f1,
        val_f1
    ])

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:02d} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"train_f1={train_f1:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    # Save "last checkpoint" every epoch
    save_full_checkpoint(
        checkpoint_path=LAST_CKPT_LOCAL,
        epoch=epoch,
        model=model_C,
        optimizer=optimizer,
        scheduler=scheduler,
        history=history,
        best_f1=best_f1,
        best_epoch=best_epoch,
        model_name=MODEL_NAME,
        y_true=y_true,
        y_pred=y_pred
    )
    copy_to_drive(LAST_CKPT_LOCAL)

    # Save best checkpoint if improved
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        save_full_checkpoint(
            checkpoint_path=BEST_CKPT_LOCAL,
            epoch=epoch,
            model=model_C,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_f1=best_f1,
            best_epoch=best_epoch,
            model_name=MODEL_NAME,
            y_true=y_true,
            y_pred=y_pred
        )
        copy_to_drive(BEST_CKPT_LOCAL)

        # Also save just weights for lightweight inference
        torch.save(model_C.state_dict(), BEST_WEIGHTS_LOCAL)
        copy_to_drive(BEST_WEIGHTS_LOCAL)

        print(f"  New best model saved at epoch {epoch} with val_macro_f1={val_f1:.4f}")

    else:
        epochs_without_improvement += 1

    # Save CSV history + metadata after every epoch
    elapsed_minutes = (time.time() - start_time) / 60
    save_training_artifacts(
        history=history,
        best_f1=best_f1,
        best_epoch=best_epoch,
        total_minutes=elapsed_minutes
    )

    # Early stopping
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after {EARLY_STOPPING_PATIENCE} epochs without improvement.")
        break

end_time = time.time()
total_minutes = (end_time - start_time) / 60

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best macro F1:", best_f1)
print("Training minutes:", total_minutes)

# Save final metadata one more time with final elapsed time
save_training_artifacts(
    history=history,
    best_f1=best_f1,
    best_epoch=best_epoch,
    total_minutes=total_minutes
)